# PKG attrition v13 · counterparty features

**Payment Knowledge Graph (PKG) · Treasury Management · Data Science**

Model 4 reads each client's own payment metrics. Customer metrics v3 and the counterparty table now let us look at
the **other end** of each relationship. This notebook builds a small monthly feature table from that view, for model 5
to add to the m4 features, and reads its signal against the m4 exit labels before any modelling.

**Output** (database `bdahd01p_dlcdi1_cdi_tm`, external parquet, partitioned by `month`)

| Table | Grain |
|---|---|
| `pkg_attr_cpty_features_monthly` | one row per TM client power id × month. `customer_pwr_id` joins to the deposit `cust_pwr_id`; all mdm_ids of one power id are one client |
| `pkg_attr_cpty_features_dictionary` | definition, type and empty-value rule of every column |

**Three families.** All point-in-time: month t uses nothing after t.

| Family | Question | Why it matters for attrition |
|---|---|---|
| Own accounts elsewhere (`own_`, `mig_`) | Do other PNC clients now pay the client, or get paid by it, through the client's own-name account at another bank? | The most direct evidence of displacement we can see, and it names the destination bank. Visible because the other end is at PNC |
| Broken relationships (`rb_`, `std_`, `nbr_`) | When a standing relationship breaks, is that counterparty still doing business with the rest of PNC? | Still active: the client moved that flow. Shrinking everywhere: contraction or market. PNC counterparties gone dark: contagion |
| Partner-adjusted change (`own_chg_`, `partner_chg_`, `padj_`) | Did the client's flow with its standing partners fall more than those partners' flow with the rest of PNC? | Difference-in-differences against the partners themselves, which removes seasonality, sector cycles and partner-side changes |

**Definitions**
- **Own-name account elsewhere:** an external entity the client itself pays or is paid by as an own-name counterparty
  (`o_self` in the v3 pair table, in any month up to t), or, with `OWN_NAME_MATCH`, an entity active this month whose
  name key equals the client's and belongs to a single client family. An entity claimed by two families is dropped, and
  so is a hub (`hub_tier` above `OWN_MAX_HUB_TIER`) unless its name is exactly the client's: the v3 self-pay match also
  accepts the first 8 characters, which on a large brand would turn every one of its payers into a "third party".
- **Third party:** a PNC customer outside the client's power id and, with `EXCLUDE_FAMILY`, outside its family
  (`master_pwr_id`). Direction is the client's view: `_in` = a third party pays the client's account elsewhere.
- **Migrated:** a third party that transacted with the client inside PNC in the same direction in the previous
  `MIG_LOOKBACK` months and now transacts with the client's account elsewhere.
- **Standing relationship:** present in at least `REC_MIN` of the previous `REC_WINDOW` months (the v3 definition);
  **broken** = standing and absent this month. Own-name accounts are left out of this family.
- **The counterparty's flow with the rest of PNC:** external counterparties from `pkg_cpty_metrics_monthly`
  (`amt_in` when the client pays it, `amt_out` when it pays the client); PNC counterparties from
  `pkg_cust_metrics_monthly_v3` (`amt_all_in` / `amt_all_out`); minus the client's own flow with it. **Observable**
  when that averages at least `OBS_MIN_AMT` a month over the baseline months; **alive** when this month is at least
  `ALIVE_FRAC` of the baseline. A PNC counterparty is **dark** when all its activity, net of the client, falls to
  `DARK_FRAC` of its baseline or less.
- **Partner-adjusted change** over observable standing relationships: `own_chg` = log of this month's flow over the
  baseline monthly flow; `partner_chg` = log of the client-weighted average partner growth (each partner's ratio capped
  at `RATIO_CAP`); `padj` = `own_chg − partner_chg`. Negative means the client fell behind its own partners.

**How to run.** Run all. Months are independent: each reads its own window of inputs, so a stop resumes where it left
off and `REBUILD_FROM` can start anywhere. A slim copy of the client side of each month's pair table is cached and
reused by the next 12 months. A first trial: `MAX_MONTHS_PER_RUN = 1`, then look at the minutes in §3.

### Pre-registered predictions, scored mechanically in §6

| # | Prediction | Confidence |
|---|---|---|
| P1 | In the latest month, at least 5% of TM client power ids have a known own-name account at another bank | 60% |
| P2 | At least 30% of broken standing relationships (by count, paid side) are observable | 50% |
| P3 | At least half of observable broken relationships are with counterparties still alive at PNC | 65% |
| P4 | Client-months with a migrated third party exit within 3 months at 3× or more the rate of those without | 55% |
| P5 | `padj_out` ranks exits with AUC at least 0.60 (lower = riskier) on the money-still-here population | 45% |
| P6 | `rb_alive_share_out` ranks exits with AUC at least 0.58 (higher = riskier) | 50% |
| P7 | Among leavers with counterparty evidence in the 6 months before exit, displacement-like outnumber contraction-like | 60% |
| P8 | At least 10% of visible leavers show a migrated third party in the 6 months before exit | 35% |
| P9 | Client-months with a dark standing PNC counterparty exit at 1.5× or more the rate of those without | 40% |

In [ ]:
# §0a  Setup. Compat shims first: Cloudera SPARK3 PySpark 3.3 predates NumPy 1.24/2.0 and pandas 2.0
import numpy as np, pandas as pd, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
for _alias, _target in {"bool": np.bool_, "object": np.object_}.items():
    if _alias not in np.__dict__:
        setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):          # removed in pandas 2.0; PySpark 3.3 createDataFrame still calls it
    pd.DataFrame.iteritems = pd.DataFrame.items
    pd.Series.iteritems = pd.Series.items
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")
warnings.filterwarnings("ignore", category=FutureWarning)

import datetime as dt, hashlib, html as _html, json, pathlib, time, uuid
import matplotlib
matplotlib.use("Agg")                               # charts go to disk only
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML

spark = (SparkSession.builder
         .appName("pkg_attr_cpty_features")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g")
         .config("spark.executor.cores", "2")
         .config("spark.executor.memoryOverhead", "3g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
         .enableHiveSupport()
         .getOrCreate())
MD = StorageLevel.MEMORY_AND_DISK
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# §0b  Config
DB               = "bdahd01p_dlcdi1_cdi_tm"
FEATURES_VERSION = "attr-cpty-v1.0"     # part of the signature: bump only when the numbers change
HISTORY_START    = "2024-01"            # first month of the PKG tables
FIRST_MONTH      = None                 # first month to build; None = HISTORY_START
LAST_MONTH       = None                 # None = latest month present in every input
MAX_MONTHS_PER_RUN = None               # e.g. 1 for a first trial, 8 to build in chunks
REBUILD_FROM     = None                 # e.g. "2026-03": rebuild that month and every later one (months are independent)

REC_WINDOW, REC_MIN = 6, 4              # standing relationship: present in ≥ REC_MIN of the previous REC_WINDOW months (= v3)
MIG_LOOKBACK     = 12                   # internal partners: seen in the previous MIG_LOOKBACK months
OBS_MIN_AMT      = 1_000                # judge a counterparty only if its monthly flow with the rest of PNC averages ≥ this ($)
ALIVE_FRAC       = 0.5                  # alive: this month's flow with the rest of PNC ≥ ALIVE_FRAC × its baseline
DARK_FRAC        = 0.1                  # dark PNC counterparty: activity net of the client ≤ DARK_FRAC × its baseline
RATIO_CAP        = 5.0                  # partner growth ratio cap, so one partner's spike can't dominate padj
OWN_NAME_MATCH   = True                 # also treat an external entity with exactly the client's name key as its own account
NAME_MIN_LEN     = 8                    # name keys shorter than this never match on name alone
OWN_MAX_HUB_TIER = 0                    # an own account above this hub_tier is kept only when its name is exactly the client's
EXCLUDE_FAMILY   = True                 # the client's own family (same master_pwr_id) doesn't count as a third party
LEGAL = r"\b(N A|INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|THE)\b"   # = v3

# §5 early signal read against the m4 exit labels
SIGNAL_READ      = True
LBL_START        = "2024-01"            # month of t = 0 in the pkg_attr panels (DATE_START of the v13 series)
HORIZON          = 3                    # exit (earlier of full exit and balance drain) within the next HORIZON months
MAX_NEG_SAMPLE   = 400_000              # stayer rows sampled for the AUCs; every exit is kept (AUC is unaffected)
OUT_DIR          = "../metrics/attrition/v13/cpty_features"     # charts and CSVs

SHUFFLE_PARTITIONS = 800
FILES = {"features": 4, "slim": 16}
CODEC = "snappy"
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))

# ---- tables
T_PAIRS    = f"{DB}.pkg_cust_pair_month_v3"
T_METRICS  = f"{DB}.pkg_cust_metrics_monthly_v3"
T_CPM      = f"{DB}.pkg_cpty_metrics_monthly"
T_CUST_DIM = f"{DB}.pkg_cust_dim"
T_LBL_MONTH, T_LBL_DIM = f"{DB}.pkg_attr_cust_month", f"{DB}.pkg_attr_cust_dim"
OUT = {"features": f"{DB}.pkg_attr_cpty_features_monthly", "dict": f"{DB}.pkg_attr_cpty_features_dictionary"}

In [ ]:
# §0c  Helpers: file system, markers, fingerprints, months, display
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))

def rm(p):
    if exists(p):
        fs_for(p).delete(JPath(p), True)

def mv(src, dst):
    """Move a finished staging folder into place (the destination is replaced)."""
    rm(dst)
    fs = fs_for(src)
    fs.mkdirs(JPath(dst).getParent())
    if not fs.rename(JPath(src), JPath(dst)):
        raise RuntimeError(f"rename failed: {src} -> {dst}")

def write_json(p, obj):
    out = fs_for(p).create(JPath(p), True)
    out.write(bytearray(json.dumps(obj, sort_keys=True, default=str).encode("utf-8")))
    out.close()

def read_json(p):
    if not exists(p):
        return None
    st = fs_for(p).open(JPath(p))
    rd = spark._jvm.java.io.BufferedReader(spark._jvm.java.io.InputStreamReader(st, "UTF-8"))
    lines, line = [], rd.readLine()
    while line is not None:
        lines.append(line); line = rd.readLine()
    rd.close()
    try:
        return json.loads("\n".join(lines))
    except ValueError:
        return None

def list_months(base):
    if not exists(base):
        return []
    return sorted(s.getPath().getName().split("=", 1)[1] for s in fs_for(base).listStatus(JPath(base))
                  if s.isDirectory() and s.getPath().getName().startswith("month="))

def path_fp(p):
    """Fingerprint of a folder: data files, bytes, latest modification (files and folders starting with _ or . skipped)."""
    if not exists(p):
        return "missing"
    it, n, size, mt = fs_for(p).listFiles(JPath(p), True), 0, 0, 0
    base = str(fs_for(p).makeQualified(JPath(p)))
    while it.hasNext():
        st = it.next()
        rel = str(st.getPath())[len(base):]
        if any(part.startswith(("_", ".")) for part in rel.split("/") if part):
            continue
        n, size, mt = n + 1, size + st.getLen(), max(mt, st.getModificationTime())
    return f"{n}:{size}:{mt}"

def table_exists(t):
    try:
        spark.table(t).schema
        return True
    except Exception:
        return False

def table_location(t):
    try:
        for r in spark.sql(f"DESCRIBE FORMATTED {t}").collect():
            if str(r[0]).strip().lower() == "location":
                return str(r[1]).strip().rstrip("/")
    except Exception:
        pass
    return f"{LOC}/{t.split('.', 1)[1]}"

def ym_idx(m):
    y, mo = map(int, m[:7].split("-"))
    return y * 12 + mo - 1

def add_months(m, k):
    i = ym_idx(m) + k
    return f"{i // 12:04d}-{i % 12 + 1:02d}"

def month_range(a, b):
    return [add_months(a, k) for k in range(ym_idx(b) - ym_idx(a) + 1)] if ym_idx(b) >= ym_idx(a) else []

def months_back(t, k0, k1):
    """Months t-k1 … t-k0 (oldest first) that are on or after HISTORY_START."""
    return [add_months(t, -k) for k in range(k1, k0 - 1, -1) if add_months(t, -k) >= HISTORY_START]

def md5(obj):
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]

def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

def pk(c, mode):
    """Power-id join key. 'strip' removes leading zeros (the deposit table and the metrics may format ids differently)."""
    s = F.trim(F.col(c).cast("string"))
    s = F.when(s == "", F.lit(None)).otherwise(s)
    return F.regexp_replace(s, r"^0+(?=.)", "") if mode == "strip" else s

def name_key(c):
    """= v3: upper-case, letters/digits/spaces only, legal suffixes removed (self-pay matching uses the same function)."""
    s = F.regexp_replace(F.upper(c), "[^A-Z0-9 ]", " ")
    s = F.trim(F.regexp_replace(F.regexp_replace(s, LEGAL, " "), r"\s+", " "))
    return F.when(F.length(s) >= 3, s)

# ---- display: one consolidated HTML output per cell -------------------------------------------------------------------
_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin:2px 0 10px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}"
        ".pkgt td:first-child,.pkgt th:first-child{text-align:left}"
        ".pkgh{margin:12px 0 3px;font-size:13px;font-weight:600}.pkgn{color:#6b6a66;margin:1px 0 6px;font-size:12px}"
        ".pkgok{color:#1e7b34;font-weight:600}.pkgbad{color:#b3261e;font-weight:600}</style>")

def _num(s):
    if s.dtype == object:
        try:
            return pd.to_numeric(s)
        except (ValueError, TypeError):
            return s
    return s

def fmt_df(df, pct=()):
    out = pd.DataFrame(index=df.index)
    for c in df.columns:
        s = df[c].astype(object) if str(c).endswith("_id") or c in ("k", "month") else _num(df[c])   # ids stay text
        if c in pct:
            out[c] = [("—" if pd.isna(v) else f"{v:.2%}") for v in s]
        elif pd.api.types.is_bool_dtype(s):
            out[c] = s.astype(str)
        elif pd.api.types.is_numeric_dtype(s):
            whole = bool((s.dropna() % 1 == 0).all())
            out[c] = [("—" if pd.isna(v) else (f"{int(v):,}" if whole else f"{v:,.3f}")) for v in s]
        else:
            out[c] = s.where(s.notna(), "—").astype(str)
    return out

def H(text):  return ("h", text)
def N(text):  return ("n", text)
def OK(text): return ("ok", text)
def BAD(text): return ("bad", text)
def TB(title, df, pct=()): return ("t", title, df, pct)

def render(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        k = b[0]
        if k in ("h", "n", "ok", "bad"):
            cls = {"h": "pkgh", "n": "pkgn", "ok": "pkgok", "bad": "pkgbad"}[k]
            parts.append(f"<div class='{cls}'>{_html.escape(str(b[1]))}</div>")
        else:
            _, title, df, pct = b
            parts.append(f"<div class='pkgh'>{_html.escape(title)} <span class='pkgn'>[{len(df):,} rows]</span></div>")
            parts.append(fmt_df(df, pct).to_html(index=False, escape=True, border=0, classes="pkgt"))
    return HTML(_CSS + "<div style=\"font-family:'IBM Plex Sans',Arial,sans-serif\">" + "".join(parts) + "</div>")

def show(*blocks):
    display(render(*blocks))

# ---- locations ---------------------------------------------------------------------------------------------------------
LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if str(r[0]).lower() == "location"][0].rstrip("/")
OUT_P = {k: f"{LOC}/{t.split('.', 1)[1]}" for k, t in OUT.items()}
CACHE  = f"{LOC}/_tmp_pkg_attr_cpty_cache"         # slim client-side pairs per month + own-account evidence (working)
STATIC = f"{CACHE}/_static"
slim_path  = lambda m: f"{CACHE}/slim/month={m}"
selfp_path = lambda m: f"{CACHE}/selfp/month={m}"
stage_path = lambda m: f"{OUT_P['features']}/_stage_month={m}"
month_dir  = lambda m: f"{OUT_P['features']}/month={m}"
MARKER = "_PKG_MONTH.json"
OUTD = pathlib.Path(OUT_DIR); OUTD.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    p = OUTD / f"{name}.png"
    fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    return str(p)
print("database location:", LOC, "| charts and CSVs:", OUTD.resolve())

## §1 Inputs and lookups

- Required columns exist and **join keys are strings** (the 2026-07 incident was an int64/str key that produced
  plausible output with every counterparty untyped).
- `pkg_cust_dim` is **one row per mdm_id**, asserted. Clients are TM customers (`is_tm`) at power-id grain; the
  family is `master_pwr_id` (the power id itself when there is none).
- **Name keys for the own-account match** are kept only when the key belongs to a single client family and is at least
  `NAME_MIN_LEN` characters long, so generic names never match.
- The **signature** covers every setting that changes the numbers and the fingerprint of `pkg_cust_dim`. Each month's
  marker also carries the fingerprints of the input months it read, so a rebuilt input month makes exactly the months
  that used it stale.

In [ ]:
# §1a  Inputs: required columns, string keys, signature
IN_LOC = {k: table_location(t) for k, t in {"pairs": T_PAIRS, "metrics": T_METRICS, "cpm": T_CPM}.items()}
month_in = lambda key, m: f"{IN_LOC[key]}/month={m}"

def schema_of(src):
    if src in IN_LOC:
        ms = list_months(IN_LOC[src])
        if not ms:
            raise RuntimeError(f"no month folders under {IN_LOC[src]}")
        return spark.read.parquet(month_in(src, ms[-1])).schema
    return spark.table(src).schema

NEED = {"pairs":   {"cust": "string", "dir": None, "scope": None, "other": "string", "amount": None, "o_fi": None, "o_self": None},
        "metrics": {"mdm_id": "string", "amt_all_in": None, "amt_all_out": None},
        "cpm":     {"ent_id": "string", "amt_in": None, "amt_out": None},
        T_CUST_DIM: {"mdm_id": "string", "customer_pwr_id": None, "master_pwr_id": None, "customer_name": None,
                     "party_type": None, "is_tm": None}}
problems, SCH = [], {}
for src, cols in NEED.items():
    try:
        sch = {f.name.lower(): f.dataType.simpleString() for f in schema_of(src).fields}
    except Exception as e:
        problems.append(f"{src}: cannot read ({str(e).splitlines()[0][:120]})"); continue
    SCH[src] = sch
    for c, typ in cols.items():
        if c not in sch:
            problems.append(f"{src}: column {c} missing")
        elif typ and sch[c] != typ:
            problems.append(f"{src}: {c} is {sch[c]}, expected {typ} (join keys must be strings)")
if problems:
    raise RuntimeError("input check failed:\n  " + "\n  ".join(problems))
CPM_OPT = {c: c in SCH["cpm"] for c in ("ent_name", "name_class", "hub_tier")}
notes = []
if OWN_NAME_MATCH and not CPM_OPT["ent_name"]:
    OWN_NAME_MATCH = False
    notes.append("pkg_cpty_metrics_monthly has no ent_name: own accounts come from self-pay evidence only")
cdq = spark.table(T_CUST_DIM).agg(F.count("*").alias("rows"), F.countDistinct("mdm_id").alias("ids")).first()
if cdq["rows"] != cdq["ids"]:
    raise RuntimeError(f"{T_CUST_DIM} must be one row per mdm_id: {cdq['rows']:,} rows, {cdq['ids']:,} ids")

AV = {k: list_months(v) for k, v in IN_LOC.items()}
FP_STATIC = {T_CUST_DIM: path_fp(table_location(T_CUST_DIM))}
CFG = {"v": FEATURES_VERSION, "start": HISTORY_START, "rec": [REC_WINDOW, REC_MIN], "mig": MIG_LOOKBACK,
       "obs": OBS_MIN_AMT, "alive": ALIVE_FRAC, "dark": DARK_FRAC, "cap": RATIO_CAP, "name": OWN_NAME_MATCH,
       "name_len": NAME_MIN_LEN, "hub": OWN_MAX_HUB_TIER, "fam": EXCLUDE_FAMILY, "legal": LEGAL,
       "cpm_opt": CPM_OPT}
SIG = md5({"cfg": CFG, "fp": FP_STATIC})
SIG_CACHE = md5({"v": "slim-v1", "fp": FP_STATIC, "start": HISTORY_START})      # slim cache: client mapping only
show(H(f"Inputs OK · signature {SIG}"),
     TB("Monthly inputs", pd.DataFrame([{"input": k, "location": IN_LOC[k], "months": len(v),
                                          "first": (v or ["—"])[0], "last": (v or ["—"])[-1]} for k, v in AV.items()])),
     TB("Optional counterparty columns", pd.DataFrame([{"column": k, "present": v} for k, v in CPM_OPT.items()])),
     *[N(x) for x in notes])

In [ ]:
# §1b  Static lookups (written once per signature): customer → client power id, family, name key; client attributes;
#      name keys that identify one client family
ST = {k: f"{STATIC}/{k}" for k in ("cmap", "pwr", "keys")}
_mk = read_json(f"{STATIC}/_PKG_STATIC.json")
if not (_mk and _mk.get("sig") == SIG and all(exists(f"{p}/_SUCCESS") for p in ST.values())):
    rm(STATIC)
    cd = (spark.table(T_CUST_DIM)
               .select(nz("mdm_id").alias("mdm_id"), nz("customer_pwr_id").alias("cpwr"), nz("master_pwr_id").alias("master"),
                       F.col("customer_name"), F.upper(nz("party_type")).alias("party"),
                       F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"))
               .withColumn("pwr", F.when(F.col("is_tm"), F.col("cpwr")))
               .withColumn("own_key", name_key(F.col("customer_name"))))
    pwr = (cd.where(F.col("pwr").isNotNull()).groupBy("pwr").agg(
               F.count("*").alias("n_mdm"), F.min("master").alias("master_pwr_id"),
               F.countDistinct("master").alias("n_master"),
               F.min(F.struct("mdm_id", "customer_name")).alias("_n"),
               F.max((F.col("party") == "O").cast("int")).alias("is_org"))
              .select("pwr", "n_mdm", "master_pwr_id", "n_master", F.col("_n.customer_name").alias("customer_name"), "is_org")
              .withColumn("family", F.coalesce("master_pwr_id", "pwr")))
    pwr.write.mode("overwrite").parquet(ST["pwr"])
    pwr = spark.read.parquet(ST["pwr"])
    (cd.join(pwr.select("pwr", "family"), "pwr", "left")
       .select("mdm_id", "pwr", "family", "party", "own_key").write.mode("overwrite").parquet(ST["cmap"]))
    kk = (cd.where(F.col("pwr").isNotNull() & (F.col("party") == "O") & F.col("own_key").isNotNull()
                   & (F.length("own_key") >= NAME_MIN_LEN))
            .select(F.col("own_key").alias("key"), "pwr").distinct().join(pwr.select("pwr", "family"), "pwr"))
    nf = kk.groupBy("key").agg(F.countDistinct("family").alias("n_fam"))
    kk.join(nf, "key").where("n_fam = 1").select("key", "pwr").write.mode("overwrite").parquet(ST["keys"])
    write_json(f"{STATIC}/_PKG_STATIC.json", {"sig": SIG, "built_at": dt.datetime.now().isoformat(timespec="seconds")})

CMAP, PWR, KEYS = (spark.read.parquet(ST[k]) for k in ("cmap", "pwr", "keys"))
TM = CMAP.where(F.col("pwr").isNotNull()).select("mdm_id", "pwr", "family")
q1 = CMAP.agg(F.count("*").alias("customers"), F.sum(F.col("pwr").isNotNull().cast("int")).alias("tm_customers")).first()
q2 = PWR.agg(F.count("*").alias("clients"), F.sum((F.col("n_mdm") > 1).cast("int")).alias("clients_2plus_mdm"),
             F.sum((F.col("n_master") > 1).cast("int")).alias("clients_2plus_masters"),
             F.countDistinct("family").alias("families"), F.sum("is_org").alias("organisations")).first()
q3 = KEYS.agg(F.countDistinct("key").alias("keys"), F.count("*").alias("key_client_pairs")).first()
show(H("Static lookups"),
     TB("Customers and clients", pd.DataFrame([{**q1.asDict(), **q2.asDict(), "name keys (one family)": q3["keys"],
                                                "key → client pairs": q3["key_client_pairs"]}])),
     N("Client = TM power id; its mdm_ids are combined. clients_2plus_masters: power ids whose mdm_ids carry different "
       "master_pwr_ids (the lowest is used as the family)."))

## §2 Column dictionary

Single source of truth for column order, type and meaning. The final select of every month casts to these types.

In [ ]:
# §2  Dictionary
D = ["in", "out"]
DEC, DBL, LNG, STR, INT = "decimal(22,2)", "double", "bigint", "string", "int"
CV = {"in": "received", "out": "paid"}
TP = {"in": "a third party pays the client's own-name account elsewhere",
      "out": "the client's own-name account elsewhere pays a third party"}
E0, ES, EN, EW = ("0 when none", "NULL when the denominator is 0", "NULL when there is nothing to name",
                  f"NULL before {REC_WINDOW} months of history")
DICT = []
def col(c, fam, d, typ, definition, empty):
    DICT.append({"column": c, "family": fam, "direction": d, "type": typ, "definition": definition, "empty_value": empty})

col("customer_pwr_id", "identity", "", STR, "TM client power id (all its mdm_ids combined); joins to the deposit cust_pwr_id", "never empty")
col("master_pwr_id", "identity", "", STR, "ultimate parent power id (lowest when the client's mdm_ids disagree)", "NULL when none")
col("customer_name", "identity", "", STR, "client name (lowest mdm_id)", "NULL when unknown")
col("n_mdm", "identity", "", INT, "mdm_ids combined into this client", "never empty")
col("window_months", "identity", "", INT, f"baseline months available (of {REC_WINDOW}) for the broken-relationship and partner families", "never empty")
col("mig_lookback_months", "identity", "", INT, f"months available (of {MIG_LOOKBACK}) for the internal-partner lookback", "never empty")
col("own_known_n", "own accounts", "", LNG, "own-name accounts at other banks known for the client as of this month", E0)
col("own_known_self_n", "own accounts", "", LNG, "of which seen in the client's own self-pay (the rest match on name only)", E0)
for d in D:
    col(f"own_ext_amt_{d}", "own accounts", d, DEC, f"dollars this month where {TP[d]}", E0)
    col(f"own_ext_n_{d}", "own accounts", d, LNG, f"distinct third parties (mdm_id) where {TP[d]}", E0)
    col(f"own_ext_fi_n_{d}", "own accounts", d, LNG, f"distinct banks of those own-name accounts, {CV[d]} side", E0)
    col(f"own_ext_top_fi_{d}", "own accounts", d, STR, f"bank holding the most of those dollars ({CV[d]} side): where the flow went", EN)
    col(f"own_ext_name_only_amt_{d}", "own accounts", d, DEC, f"part of own_ext_amt_{d} through accounts matched on name only", E0)
    col(f"int_partners_n_{d}", "own accounts", d, LNG, f"distinct PNC customers the client {CV[d]} inside PNC in the previous {MIG_LOOKBACK} months", E0)
    col(f"mig_n_{d}", "own accounts", d, LNG, f"migrated third parties: in int_partners_n_{d} and now transacting with the client's own-name account elsewhere", E0)
    col(f"mig_amt_{d}", "own accounts", d, DEC, f"dollars this month of the migrated third parties with the client's own-name accounts", E0)
    col(f"mig_share_{d}", "own accounts", d, DBL, f"mig_n_{d} / int_partners_n_{d}", ES)
for d in D:
    col(f"std_n_{d}", "broken relationships", d, LNG, f"standing relationships, {CV[d]} side (≥ {REC_MIN} of the previous {REC_WINDOW} months; own-name accounts excluded)", EW)
    col(f"std_amt_{d}", "broken relationships", d, DEC, "baseline monthly dollars of the standing relationships (mean of the previous months, absent months = 0)", EW)
    col(f"std_obs_amt_share_{d}", "broken relationships", d, DBL, "share of std_amt with counterparties observable at PNC (their flow with the rest of PNC averages ≥ OBS_MIN_AMT)", EW + "; " + ES)
    col(f"rb_n_{d}", "broken relationships", d, LNG, "standing relationships absent this month", EW)
    col(f"rb_amt_{d}", "broken relationships", d, DEC, "baseline monthly dollars of the broken relationships", EW)
    col(f"rb_obs_n_{d}", "broken relationships", d, LNG, "broken relationships whose counterparty is observable", EW)
    col(f"rb_alive_n_{d}", "broken relationships", d, LNG, f"of those, counterparty still alive: this month's flow with the rest of PNC ≥ {ALIVE_FRAC:.0%} of its baseline (the client moved the flow)", EW)
    col(f"rb_shrunk_n_{d}", "broken relationships", d, LNG, "rb_obs_n − rb_alive_n: counterparty shrinking at PNC too (contraction or market)", EW)
    col(f"rb_alive_share_{d}", "broken relationships", d, DBL, f"rb_alive_n_{d} / rb_obs_n_{d}", EW + "; " + ES)
    col(f"rb_alive_amt_share_{d}", "broken relationships", d, DBL, "baseline dollars of alive broken / baseline dollars of observable broken", EW + "; " + ES)
    col(f"own_chg_{d}", "partner-adjusted", d, DBL, "log((this month + 1) / (baseline + 1)) of the client's dollars with its observable standing partners", EW + "; NULL without observable partners")
    col(f"partner_chg_{d}", "partner-adjusted", d, DBL, f"log of the client-weighted mean of the partners' growth with the rest of PNC, ((now + 1) / (baseline + 1)) capped at {RATIO_CAP}", EW + "; NULL without observable partners")
    col(f"padj_{d}", "partner-adjusted", d, DBL, f"own_chg_{d} − partner_chg_{d}: below 0 = the client fell behind its own partners", EW + "; NULL without observable partners")
col("nbr_std_n", "broken relationships", "", LNG, f"standing PNC counterparties (either direction, ≥ {REC_MIN} of {REC_WINDOW} months)", EW)
col("nbr_dark_n", "broken relationships", "", LNG, f"of those, gone dark: activity net of the client ≤ {DARK_FRAC:.0%} of its baseline (baseline ≥ OBS_MIN_AMT)", EW)
col("nbr_dark_amt_share", "broken relationships", "", DBL, "share of the client's baseline dollars with standing PNC counterparties that went dark (contagion)", EW + "; " + ES)

OUT_COLS = [r["column"] for r in DICT]
TYPES = {r["column"]: r["type"] for r in DICT}
BD_COLS = [r["column"] for r in DICT if r["empty_value"].startswith(EW)]
DICT_DF = pd.DataFrame(DICT)
DICT_DF.insert(0, "table", OUT["features"])
assert not DICT_DF.duplicated("column").any()
DICT_DF = pd.concat([DICT_DF, pd.DataFrame([{"table": OUT["features"], "column": "month", "family": "partition", "direction": "",
                                             "type": STR, "definition": "month YYYY-MM", "empty_value": "never empty"}])],
                    ignore_index=True)
spark.createDataFrame(DICT_DF.astype(str)).write.mode("overwrite").option("compression", CODEC).parquet(OUT_P["dict"])
spark.sql(f"DROP TABLE IF EXISTS {OUT['dict']}")
spark.sql(f"CREATE EXTERNAL TABLE {OUT['dict']} (" + ", ".join(f"`{c}` string" for c in DICT_DF.columns)
          + f") STORED AS PARQUET LOCATION '{OUT_P['dict']}'")
spark.catalog.refreshTable(OUT["dict"])
show(H(f"Dictionary written to {OUT['dict']}"),
     TB("Columns by family", DICT_DF.groupby("family", sort=False).column.count().reset_index()))

## §3 Monthly build

Each month t:

1. **Slim cache** (per input month, reused by the next 12 months): the client side of the v3 pair table at power-id
   grain, TM clients only. Flows between two mdm_ids of the same power id are the client's own money and are dropped.
   Reconciled to the pair table when written. Own-account evidence (`o_self` pairs) is extracted alongside.
2. **Own-account registry** as of t: self-pay evidence up to t, plus name matches among entities active in t; entities
   claimed by two families dropped.
3. **Own accounts elsewhere:** third parties' month-t pairs with those entities, matched to the client's internal
   partners of the previous 12 months.
4. **Broken relationships and partner change:** the client's pairs over t and the 6 previous months; the counterparties'
   flow with the rest of PNC from the counterparty table (external) or the customer metrics (PNC customers).
5. **Assemble** one row per client, check invariants and the own-account dollars, stage, move into place, write the
   month marker.

In [ ]:
# §3a  Step functions
_FPC, _SLIM_OK, _SELFP_OK, _BUILT = {}, set(), set(), []   # per-run memo: input fingerprints, verified caches, caches built

def fp_of(p):
    if p not in _FPC:
        _FPC[p] = path_fp(p)
    return _FPC[p]

def get_slim(m):
    """Client side of the pair table for month m at power-id grain (cached; rebuilt when the pair month changes)."""
    if m in _SLIM_OK:
        return spark.read.parquet(slim_path(m))
    p, fp = slim_path(m), fp_of(month_in("pairs", m))
    mk = read_json(f"{p}/_PKG_CACHE.json")
    if mk and mk.get("sig") == SIG_CACHE and mk.get("fp") == fp and exists(f"{p}/_SUCCESS") and exists(f"{selfp_path(m)}/_SUCCESS"):
        _SLIM_OK.add(m)
        return spark.read.parquet(p)
    rm(p); rm(selfp_path(m)); _SELFP_OK.discard(m)
    pr = (spark.read.parquet(month_in("pairs", m)).where(F.col("scope").isin("int", "ext"))
               .select("cust", "dir", "scope", "other", "amount",
                       F.coalesce(F.col("o_self").cast("int"), F.lit(0)).alias("o_self"))
               .join(TM.select(F.col("mdm_id").alias("cust"), "pwr"), "cust", "inner").persist(MD))
    pi = pr.where(F.col("scope") == "int").join(TM.select(F.col("mdm_id").alias("other"), F.col("pwr").alias("o_pwr")), "other", "left")
    intra = F.coalesce(F.col("o_pwr") == F.col("pwr"), F.lit(False))
    tot = float(pr.agg(F.sum("amount")).first()[0] or 0)
    tot_intra = float(pi.where(intra).agg(F.sum("amount")).first()[0] or 0)
    keep = pi.where(~intra).drop("o_pwr").unionByName(pr.where(F.col("scope") == "ext"))
    (keep.groupBy("pwr", "dir", "scope", "other")
         .agg(F.sum(F.col("amount").cast("double")).alias("a"), F.max("o_self").alias("o_self"))
         .repartition(FILES["slim"], "pwr").write.mode("overwrite").option("compression", CODEC).parquet(p))
    pr.unpersist()
    s = spark.read.parquet(p)
    got = float(s.agg(F.sum("a")).first()[0] or 0)
    if abs(got - (tot - tot_intra)) > max(1.0, 1e-9 * abs(tot)):
        rm(p)
        raise AssertionError(f"slim cache {m} does not reconcile: pairs {tot:,.2f} − own-power-id {tot_intra:,.2f} vs slim {got:,.2f}")
    s.where("scope = 'ext' AND o_self = 1").select("pwr", "other").distinct().write.mode("overwrite").parquet(selfp_path(m))
    write_json(f"{selfp_path(m)}/_PKG_CACHE.json", {"sig": SIG_CACHE, "fp": fp})
    write_json(f"{p}/_PKG_CACHE.json", {"sig": SIG_CACHE, "fp": fp, "pairs_usd": tot, "own_pwr_usd": tot_intra})
    _SLIM_OK.add(m); _SELFP_OK.add(m); _BUILT.append(m)
    return s

def ensure_selfp(m):
    """Own-account evidence of month m. Kept for every month (small); the slim cache is rebuilt only if this is stale."""
    if m in _SELFP_OK:
        return
    mk = read_json(f"{selfp_path(m)}/_PKG_CACHE.json")
    if mk and mk.get("sig") == SIG_CACHE and mk.get("fp") == fp_of(month_in("pairs", m)) and exists(f"{selfp_path(m)}/_SUCCESS"):
        _SELFP_OK.add(m)
        return
    rm(f"{slim_path(m)}/_PKG_CACHE.json"); _SLIM_OK.discard(m)
    get_slim(m)

def own_registry(t):
    """Own-name accounts at other banks as of month t: self-pay evidence up to t, name matches among entities active in t.
    hub_drop: a hub entity (hub_tier > OWN_MAX_HUB_TIER) whose name is not exactly the client's, typically a first-8-
    characters self-pay match on a large brand. n_fam: families claiming the entity (only 1 is kept)."""
    ms = month_range(HISTORY_START, t)
    for mm in ms:
        ensure_selfp(mm)
    reg = spark.read.parquet(*[selfp_path(mm) for mm in ms]).distinct().withColumn("self_ev", F.lit(1))
    ent = spark.read.parquet(month_in("cpm", t)).select(
        F.col("ent_id").alias("other"),
        (name_key(F.col("ent_name")) if CPM_OPT["ent_name"] else F.lit(None).cast("string")).alias("ekey"),
        ((F.col("name_class") == "business") if CPM_OPT["name_class"] else F.lit(True)).alias("biz"),
        (F.coalesce(F.col("hub_tier").cast("int"), F.lit(0)) if CPM_OPT["hub_tier"] else F.lit(0)).alias("tier"))
    if OWN_NAME_MATCH:
        nm = (ent.where(F.coalesce(F.col("biz"), F.lit(False)) & F.col("ekey").isNotNull())
                 .join(KEYS.withColumnRenamed("key", "ekey"), "ekey", "inner")
                 .select("pwr", "other").distinct().withColumn("name_ev", F.lit(1)))
        reg = reg.join(nm, ["pwr", "other"], "full")
    else:
        reg = reg.withColumn("name_ev", F.lit(None).cast("int"))
    own = (CMAP.where(F.col("pwr").isNotNull() & F.col("own_key").isNotNull())
               .select("pwr", F.col("own_key").alias("ekey")).distinct().withColumn("exact", F.lit(1)))
    reg = (reg.select("pwr", "other", F.coalesce("self_ev", F.lit(0)).alias("self_ev"), F.coalesce("name_ev", F.lit(0)).alias("name_ev"))
              .join(ent.select("other", "ekey", "tier"), "other", "left")
              .join(own, ["pwr", "ekey"], "left")
              .withColumn("exact", F.coalesce("exact", F.lit(0))).withColumn("tier", F.coalesce("tier", F.lit(0)))
              .withColumn("hub_drop", (F.col("exact") == 0) & (F.col("tier") > OWN_MAX_HUB_TIER))
              .join(PWR.select("pwr", "family"), "pwr", "left"))
    nf = reg.where(~F.col("hub_drop")).groupBy("other").agg(F.countDistinct(F.coalesce("family", "pwr")).alias("n_fam"))
    return reg.join(nf, "other", "left")

A_LONG = ["own_ext_amt", "own_ext_n", "own_ext_name_only_amt", "own_ext_fi_n", "own_ext_top_fi", "int_partners_n",
          "mig_n", "mig_amt"]

def fam_own(t, reg):
    """Third parties' month-t flows with the clients' own-name accounts; migrated internal partners."""
    pr = (spark.read.parquet(month_in("pairs", t)).where(F.col("scope") == "ext")
               .select(F.col("cust").alias("a_mdm"), "dir", "other", F.col("amount").cast("double").alias("amt"), "o_fi"))
    r = (pr.join(reg.select("pwr", "other", "self_ev"), "other", "inner")
           .join(CMAP.select(F.col("mdm_id").alias("a_mdm"), F.col("pwr").alias("a_pwr"), F.col("family").alias("a_family")), "a_mdm", "left")
           .join(PWR.select("pwr", F.col("family").alias("c_family")), "pwr", "left")
           .where(~F.coalesce(F.col("a_pwr") == F.col("pwr"), F.lit(False))))          # the client's own mdm_ids: self-pay, not a third party
    if EXCLUDE_FAMILY:
        r = r.where(~F.coalesce(F.col("a_family") == F.col("c_family"), F.lit(False)))
    r = r.withColumn("cdir", F.when(F.col("dir") == "out", "in").otherwise("out")).persist(MD)   # client's view
    K = ["pwr", "cdir"]
    base = r.groupBy(*K).agg(F.sum("amt").alias("own_ext_amt"), F.countDistinct("a_mdm").alias("own_ext_n"),
                             F.sum(F.when(F.col("self_ev") == 0, F.col("amt")).otherwise(0.0)).alias("own_ext_name_only_amt"))
    fi = (r.where(F.col("o_fi").isNotNull()).groupBy(*K, "o_fi").agg(F.sum("amt").alias("fa"))
           .groupBy(*K).agg(F.count("*").alias("own_ext_fi_n"), F.max(F.struct("fa", "o_fi")).alias("_t"))   # ties: name order
           .select(*K, "own_ext_fi_n", F.col("_t.o_fi").alias("own_ext_top_fi")))
    L = base.join(fi, K, "full")
    lb = months_back(t, 1, MIG_LOOKBACK)
    part = None
    if lb:
        for mm in lb:
            get_slim(mm)
        part = (spark.read.parquet(*[slim_path(mm) for mm in lb]).where(F.col("scope") == "int")
                     .select("pwr", F.col("dir").alias("cdir"), F.col("other").alias("a_mdm")).distinct().persist(MD))
        pn = part.groupBy(*K).agg(F.count("*").alias("int_partners_n"))
        mig = (r.join(part, K + ["a_mdm"], "inner").groupBy(*K)
                .agg(F.countDistinct("a_mdm").alias("mig_n"), F.sum("amt").alias("mig_amt")))
        L = L.join(pn, K, "full").join(mig, K, "full")
    for c in A_LONG:
        if c not in L.columns:
            L = L.withColumn(c, F.lit(None))
    W = L.groupBy("pwr").agg(*[F.max(F.when(F.col("cdir") == d, F.col(c))).alias(f"{c}_{d}") for c in A_LONG for d in D])
    known = reg.groupBy("pwr").agg(F.count("*").alias("own_known_n"), F.sum("self_ev").alias("own_known_self_n"))
    return W.join(known, "pwr", "full"), r, part

def pivot_totals(key, id_col, c_in, c_out, js, keys):
    """Counterparty totals by month offset j (0 = t): ti{j} = received from PNC customers, to{j} = paid to them."""
    T = None
    for j, mm in js:
        x = spark.read.parquet(month_in(key, mm)).select(F.col(id_col).alias("other"), F.col(c_in).cast("double").alias("ti"),
                                                          F.col(c_out).cast("double").alias("to"), F.lit(j).alias("k"))
        T = x if T is None else T.unionByName(x)
    T = T.join(keys, "other", "inner")
    J = [j for j, _ in js]
    return T.groupBy("other").agg(*[F.sum(F.when(F.col("k") == j, F.col("ti"))).alias(f"ti{j}") for j in J],
                                  *[F.sum(F.when(F.col("k") == j, F.col("to"))).alias(f"to{j}") for j in J])

def fam_bd(t):
    """Broken standing relationships (alive vs shrinking counterparties), partner-adjusted change, dark PNC neighbours."""
    js = [(j, add_months(t, -j)) for j in range(REC_WINDOW + 1) if add_months(t, -j) >= HISTORY_START]
    wm = sum(1 for j, _ in js if j >= 1)
    S = None
    for j, mm in js:
        s = get_slim(mm).select("pwr", "dir", "scope", "other", F.col("a").alias("x"), "o_self", F.lit(j).alias("k"))
        S = s if S is None else S.unionByName(s)
    U = S.select("pwr").distinct()                      # every client with any flow in the window, own accounts included
    G = (S.where(F.col("o_self") == 0).groupBy("pwr", "dir", "scope", "other")
          .agg(*[F.sum(F.when(F.col("k") == j, F.col("x"))).alias(f"a{j}") for j, _ in js]))
    if wm < REC_WINDOW:
        return U, None, None, wm, None
    JW = list(range(1, REC_WINDOW + 1))
    for j in range(REC_WINDOW + 1):
        G = G.withColumn(f"a{j}", F.coalesce(F.col(f"a{j}"), F.lit(0.0)))
    npres = lambda p: sum((F.col(f"{p}{j}") > 0).cast("int") for j in JW)
    meanw = lambda p: sum(F.col(f"{p}{j}") for j in JW) / F.lit(float(REC_WINDOW))
    G = G.withColumn("npres", npres("a")).withColumn("base", meanw("a")).persist(MD)
    STD = G.where(F.col("npres") >= REC_MIN)
    GI = (G.where(F.col("scope") == "int").groupBy("pwr", "other")
           .agg(*[F.sum(f"a{j}").alias(f"c{j}") for j in range(REC_WINDOW + 1)]))
    GI = GI.withColumn("npres", npres("c")).withColumn("base", meanw("c")).where(F.col("npres") >= REC_MIN).persist(MD)
    keys_e = STD.where(F.col("scope") == "ext").select("other").distinct()
    keys_i = STD.where(F.col("scope") == "int").select("other").unionByName(GI.select("other")).distinct()
    TE = pivot_totals("cpm", "ent_id", "amt_in", "amt_out", js, keys_e)
    TI = pivot_totals("metrics", "mdm_id", "amt_all_in", "amt_all_out", js, keys_i).persist(MD)
    R = (STD.where(F.col("scope") == "ext").join(TE, "other", "left")
            .unionByName(STD.where(F.col("scope") == "int").join(TI, "other", "left")))
    # the counterparty's flow with the rest of PNC, in the direction that matches the relationship:
    # the client pays it (out) → it receives (ti); it pays the client (in) → it pays (to). Net of the client's own flow.
    for j in range(REC_WINDOW + 1):
        tj = F.coalesce(F.when(F.col("dir") == "out", F.col(f"ti{j}")).otherwise(F.col(f"to{j}")), F.lit(0.0))
        R = R.withColumn(f"p{j}", F.greatest(tj - F.col(f"a{j}"), F.lit(0.0)))
    R = (R.withColumn("PB", meanw("p")).withColumn("Pt", F.col("p0"))
          .withColumn("obs", F.col("PB") >= OBS_MIN_AMT)
          .withColumn("alive", F.col("obs") & (F.col("Pt") >= F.lit(ALIVE_FRAC) * F.col("PB")))
          .withColumn("broken", F.col("a0") <= 0)
          .withColumn("ratio", F.least((F.col("Pt") + 1.0) / (F.col("PB") + 1.0), F.lit(RATIO_CAP))))
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0)).cast("long")
    sm = lambda cond, x: F.sum(F.when(cond, x).otherwise(0.0))
    ob, br, al = F.col("obs"), F.col("broken"), F.col("alive")
    BL = R.groupBy("pwr", "dir").agg(
        F.count("*").alias("std_n"), F.sum("base").alias("std_amt"), sm(ob, F.col("base")).alias("_obs_base"),
        cnt(br).alias("rb_n"), sm(br, F.col("base")).alias("rb_amt"), cnt(br & ob).alias("rb_obs_n"),
        cnt(br & al).alias("rb_alive_n"), sm(br & ob, F.col("base")).alias("_rb_obs_amt"),
        sm(br & al, F.col("base")).alias("_rb_alive_amt"), sm(ob, F.col("a0")).alias("_obs_now"),
        sm(ob, F.col("base") * F.col("ratio")).alias("_part_w"))
    BW = BL.groupBy("pwr").agg(*[F.max(F.when(F.col("dir") == d, F.col(c))).alias(f"{c}_{d}")
                                 for c in BL.columns if c not in ("pwr", "dir") for d in D])
    # PNC counterparties gone dark: all their activity net of the client (both directions)
    NI = GI.join(TI, "other", "left")
    for j in range(REC_WINDOW + 1):
        NI = NI.withColumn(f"o{j}", F.greatest(F.coalesce(F.col(f"ti{j}"), F.lit(0.0)) + F.coalesce(F.col(f"to{j}"), F.lit(0.0))
                                               - F.col(f"c{j}"), F.lit(0.0)))
    NI = (NI.withColumn("OB", meanw("o"))
            .withColumn("dark", (F.col("OB") >= OBS_MIN_AMT) & (F.col("o0") <= F.lit(DARK_FRAC) * F.col("OB"))))
    NW = NI.groupBy("pwr").agg(F.count("*").alias("nbr_std_n"), cnt(F.col("dark")).alias("nbr_dark_n"),
                               F.sum("base").alias("_nbr_base"), sm(F.col("dark"), F.col("base")).alias("_nbr_dark_base"))
    return U, BW, NW, wm, [G, GI, TI]

def assemble(U, AW, BW, NW, wm, lbm):
    X = U.join(AW, "pwr", "left")
    if BW is not None:
        X = X.join(BW, "pwr", "left").join(NW, "pwr", "left")
    X = X.join(PWR.select("pwr", "master_pwr_id", "customer_name", "n_mdm"), "pwr", "left")
    z = lambda c: F.coalesce(F.col(c), F.lit(0))
    zd = lambda c: F.coalesce(F.col(c).cast("double"), F.lit(0.0))
    sd = lambda num, den: F.when(den > 0, num / den)
    X = X.withColumn("own_known_n", z("own_known_n")).withColumn("own_known_self_n", z("own_known_self_n"))
    for d in D:
        for c in ["own_ext_amt", "own_ext_n", "own_ext_name_only_amt", "own_ext_fi_n", "int_partners_n", "mig_n", "mig_amt"]:
            X = X.withColumn(f"{c}_{d}", z(f"{c}_{d}"))
        X = X.withColumn(f"mig_share_{d}", sd(zd(f"mig_n_{d}"), zd(f"int_partners_n_{d}")))
        if BW is not None:
            for c in ["std_n", "std_amt", "rb_n", "rb_amt", "rb_obs_n", "rb_alive_n", "_obs_base", "_rb_obs_amt",
                      "_rb_alive_amt", "_obs_now", "_part_w"]:
                X = X.withColumn(f"{c}_{d}", z(f"{c}_{d}"))
            ob = zd(f"_obs_base_{d}")
            X = (X.withColumn(f"rb_shrunk_n_{d}", F.col(f"rb_obs_n_{d}") - F.col(f"rb_alive_n_{d}"))
                  .withColumn(f"rb_alive_share_{d}", sd(zd(f"rb_alive_n_{d}"), zd(f"rb_obs_n_{d}")))
                  .withColumn(f"rb_alive_amt_share_{d}", sd(zd(f"_rb_alive_amt_{d}"), zd(f"_rb_obs_amt_{d}")))
                  .withColumn(f"std_obs_amt_share_{d}", sd(ob, zd(f"std_amt_{d}")))
                  .withColumn(f"own_chg_{d}", F.when(ob > 0, F.log((zd(f"_obs_now_{d}") + 1.0) / (ob + 1.0))))
                  .withColumn(f"partner_chg_{d}", F.when(ob > 0, F.log(zd(f"_part_w_{d}") / ob)))
                  .withColumn(f"padj_{d}", F.col(f"own_chg_{d}") - F.col(f"partner_chg_{d}")))
    if BW is not None:
        X = (X.withColumn("nbr_std_n", z("nbr_std_n")).withColumn("nbr_dark_n", z("nbr_dark_n"))
              .withColumn("nbr_dark_amt_share", sd(zd("_nbr_dark_base"), zd("_nbr_base"))))
    else:
        for c in BD_COLS:
            X = X.withColumn(c, F.lit(None))
    X = (X.withColumnRenamed("pwr", "customer_pwr_id")
          .withColumn("window_months", F.lit(wm)).withColumn("mig_lookback_months", F.lit(lbm)))
    return X.select(*[F.col(c).cast(TYPES[c]).alias(c) for c in OUT_COLS])

INV = {
    "rb_alive ≤ rb_obs ≤ rb ≤ standing": lambda d: ((F.col(f"rb_alive_n_{d}") > F.col(f"rb_obs_n_{d}"))
                                                    | (F.col(f"rb_obs_n_{d}") > F.col(f"rb_n_{d}")) | (F.col(f"rb_n_{d}") > F.col(f"std_n_{d}"))),
    "mig ≤ own_ext and ≤ internal partners": lambda d: ((F.col(f"mig_n_{d}") > F.col(f"own_ext_n_{d}"))
                                                        | (F.col(f"mig_n_{d}") > F.col(f"int_partners_n_{d}"))),
    "shares within [0, 1]": lambda d: (~F.col(f"mig_share_{d}").between(0, 1) | ~F.col(f"rb_alive_share_{d}").between(0, 1)
                                      | ~F.col(f"rb_alive_amt_share_{d}").between(0, 1) | ~F.col(f"std_obs_amt_share_{d}").between(0, 1)),
    "amounts ≥ 0": lambda d: ((F.col(f"own_ext_amt_{d}") < 0) | (F.col(f"rb_amt_{d}") < 0) | (F.col(f"std_amt_{d}") < 0)
                              | (F.col(f"own_ext_name_only_amt_{d}") > F.col(f"own_ext_amt_{d}") + 0.01)),
}

In [ ]:
# §3b  Run: resume, stage, check, publish
def ensure_table(schema):
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in schema.fields if f.name != "month")
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {OUT['features']} ({cols}) PARTITIONED BY (month STRING) "
              f"STORED AS PARQUET LOCATION '{OUT_P['features']}'")

def add_partition(m):
    spark.sql(f"ALTER TABLE {OUT['features']} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{month_dir(m)}'")

def reregister():
    """Drop + re-create the table definition from the newest month's schema and re-attach every month on disk."""
    months = list_months(OUT_P["features"])
    if not months:
        return
    schema = spark.read.parquet(month_dir(months[-1])).schema
    spark.sql(f"DROP TABLE IF EXISTS {OUT['features']}")
    ensure_table(schema)
    for mm in months:
        add_partition(mm)
    spark.catalog.refreshTable(OUT["features"])

def inputs_fp(t):
    """Fingerprints of every input month month t reads."""
    fps = {f"pairs/{mm}": fp_of(month_in("pairs", mm)) for mm in month_range(HISTORY_START, t)}
    for k in ("cpm", "metrics"):
        for mm in months_back(t, 0, REC_WINDOW):
            fps[f"{k}/{mm}"] = fp_of(month_in(k, mm))
    return md5(fps)

def marker(m):
    return read_json(f"{month_dir(m)}/{MARKER}")

def build_month(t, fp, log):
    t_month, cleanup = time.time(), []
    del _BUILT[:]
    t0 = time.time()
    reg_all = own_registry(t).persist(MD); cleanup.append(reg_all)
    rq = reg_all.agg(F.countDistinct(F.when(F.col("n_fam") == 1, F.col("other"))).alias("e"),
                     F.countDistinct(F.when(F.col("n_fam") > 1, F.col("other"))).alias("amb"),
                     F.countDistinct(F.when(F.col("hub_drop"), F.col("other"))).alias("hub")).first()
    reg = reg_all.where((F.col("n_fam") == 1) & ~F.col("hub_drop"))
    log["steps"]["registry"] = round((time.time() - t0) / 60, 2)

    t0 = time.time()
    AW, R, part = fam_own(t, reg)
    cleanup += [R] + ([part] if part is not None else [])
    log["steps"]["own"] = round((time.time() - t0) / 60, 2)
    t0 = time.time()
    U, BW, NW, wm, pers = fam_bd(t)
    cleanup += pers or []
    log["steps"]["broken"] = round((time.time() - t0) / 60, 2)

    t0 = time.time()
    U = U.unionByName(AW.select("pwr")).distinct()
    lbm = len(months_back(t, 1, MIG_LOOKBACK))
    X = assemble(U, AW, BW, NW, wm, lbm)
    X.repartition(FILES["features"]).write.mode("overwrite").option("compression", CODEC).parquet(stage_path(t))
    st = spark.read.parquet(stage_path(t))
    g = st.agg(F.count("*").alias("rows"), F.countDistinct("customer_pwr_id").alias("distinct"),
               F.sum(F.col("own_ext_amt_in") + F.col("own_ext_amt_out")).cast("double").alias("own_usd"),
               *[F.sum(F.coalesce(f(d), F.lit(False)).cast("int")).alias(f"inv{i}_{d}") for i, f in enumerate(INV.values()) for d in D],
               F.sum((F.col("own_known_n") > 0).cast("int")).alias("own_known"),
               F.sum(((F.col("own_ext_n_in") + F.col("own_ext_n_out")) > 0).cast("int")).alias("own_flow"),
               F.sum(((F.col("mig_n_in") + F.col("mig_n_out")) > 0).cast("int")).alias("mig"),
               F.sum("rb_n_out").alias("rb_out"), F.sum("rb_obs_n_out").alias("rb_obs_out"),
               F.sum("rb_alive_n_out").alias("rb_alive_out"), F.sum((F.col("nbr_dark_n") > 0).cast("int")).alias("dark")).first()
    r_usd = float(R.agg(F.sum("amt")).first()[0] or 0)
    checks = {"one row per client": g["rows"] == g["distinct"],
              "own-account dollars = third-party pairs": abs((g["own_usd"] or 0) - r_usd) <= max(1.0, 1e-9 * abs(r_usd))}
    for i, name in enumerate(INV):
        checks[name] = all((g[f"inv{i}_{d}"] or 0) == 0 for d in D)
    log["steps"]["assemble"] = round((time.time() - t0) / 60, 2)
    log.update({"clients": g["rows"], "own_known": g["own_known"], "own_flow": g["own_flow"], "migrated": g["mig"],
                "own_usd_m": (g["own_usd"] or 0) / 1e6, "broken_out": g["rb_out"], "observable_out": g["rb_obs_out"],
                "alive_out": g["rb_alive_out"], "dark_nbr": g["dark"], "own_entities": rq["e"], "ambiguous_dropped": rq["amb"], "hub_dropped": rq["hub"],
                "window_months": wm, "caches_built": len(_BUILT)})
    failed = [k for k, ok in checks.items() if not ok]
    for x in cleanup:
        x.unpersist()
    if failed:
        rm(stage_path(t))
        raise AssertionError(f"{t}: checks failed: {failed}. Nothing was published; the month will be redone.")
    mv(stage_path(t), month_dir(t))
    ensure_table(spark.read.parquet(month_dir(t)).schema)
    add_partition(t)
    write_json(f"{month_dir(t)}/{MARKER}", {"sig": SIG, "fp": fp, "version": FEATURES_VERSION, "rows": g["rows"],
                                            "built_at": dt.datetime.now().isoformat(timespec="seconds")})
    # slim months older than the lookback of the next month are no longer needed (rebuilt on demand if they are)
    keep_from = add_months(t, 1 - MIG_LOOKBACK)
    for mm in list_months(f"{CACHE}/slim"):
        if mm < keep_from:
            rm(slim_path(mm)); _SLIM_OK.discard(mm)
    log["minutes"] = round((time.time() - t_month) / 60, 2)
    with open("pkg_attr_cpty_features_runlog.csv", "a") as f:
        f.write(f"{dt.datetime.now().isoformat(timespec='seconds')},{t},{log['minutes']},{g['rows']},{SIG}\n")

# ---- which months: every month whose inputs exist, then done / stale ------------------------------------------------
_FPC.clear(); _SLIM_OK.clear(); _SELFP_OK.clear()
cands = sorted(set(AV["pairs"]) & set(AV["cpm"]) & set(AV["metrics"]))
last = LAST_MONTH or (cands[-1] if cands else None)
first = FIRST_MONTH or HISTORY_START
RUN_MONTHS = [m for m in month_range(first, last)] if last else []
need_missing = sorted({mm for t in RUN_MONTHS for mm in (month_range(HISTORY_START, t)) if mm not in set(AV["pairs"])}
                      | {mm for t in RUN_MONTHS for mm in months_back(t, 0, REC_WINDOW) if mm not in set(AV["cpm"]) or mm not in set(AV["metrics"])})
if need_missing:
    raise RuntimeError(f"input months missing: {need_missing}")
if REBUILD_FROM:
    for m in [x for x in RUN_MONTHS if x >= REBUILD_FROM]:
        rm(f"{month_dir(m)}/{MARKER}")
    print(f"REBUILD_FROM = {REBUILD_FROM}: markers removed. Set REBUILD_FROM = None after this run.")
status, FPS = [], {}
for m in RUN_MONTHS:
    mk, FPS[m] = marker(m), inputs_fp(m)
    why = ("done" if mk and mk.get("sig") == SIG and mk.get("fp") == FPS[m] and exists(f"{month_dir(m)}/_SUCCESS") else
           "no marker" if not mk else "settings or customer dim changed" if mk.get("sig") != SIG else "input months changed")
    status.append({"month": m, "status": why})
todo = [s["month"] for s in status if s["status"] != "done"]
if MAX_MONTHS_PER_RUN:
    todo = todo[:MAX_MONTHS_PER_RUN]
st_df = (pd.DataFrame(status).groupby("status").month.agg(["count", "min", "max"]).reset_index()
         if status else pd.DataFrame(columns=["status", "count", "min", "max"]))

MONTH_LOG = []
def _rows():
    return pd.DataFrame([{**{k: v for k, v in r.items() if k != "steps"}, **{f"min_{k}": v for k, v in r["steps"].items()}}
                         for r in MONTH_LOG])
handle = display(render(H(f"Signature {SIG} · {len(RUN_MONTHS)} months {RUN_MONTHS[0] if RUN_MONTHS else '—'} → "
                          f"{RUN_MONTHS[-1] if RUN_MONTHS else '—'} · to build now: {len(todo)}"), TB("Month status", st_df)),
                 display_id=True)
for m in todo:
    log = {"month": m, "steps": {}}
    try:
        build_month(m, FPS[m], log)
    except Exception as e:
        log["error"] = str(e).splitlines()[0][:220]
        MONTH_LOG.append(log)
        handle.update(render(H(f"Stopped at {m}: rerun this cell to resume from {m}"), BAD(log["error"]), TB("This run", _rows())))
        raise
    MONTH_LOG.append(log)
    handle.update(render(H(f"Signature {SIG} · built {len(MONTH_LOG)} of {len(todo)} this run · last {m}"),
                         TB("Month status before this run", st_df), TB("This run", _rows())))
reregister()
handle.update(render(H(f"Signature {SIG} · built {len(MONTH_LOG)} of {len(todo)} this run · all checks passed"),
                     TB("Month status before this run", st_df),
                     TB("This run", _rows()) if MONTH_LOG else N("Nothing to build: every month is done for this signature."),
                     N("clients = rows; own_known / own_flow / migrated = clients with a known own account / third-party flow "
                       "through it / a migrated partner; broken/observable/alive_out = relationships, paid side. "
                       "Table re-created: run INVALIDATE METADATA in Impala (§7).")))

## §4 QA

- **Latest month:** coverage of each family (clients and baseline dollars), distributions, the clients with the most
  migrated partners and the banks their flows went to. These are the cases to sanity-check by name.
- **All months:** the monthly series of the main features, to catch coverage shifts (the possible 2026-06 coverage
  change and the 2026-07/08 dollar jump). Chart saved to disk.

In [ ]:
# §4  QA
RES, blocks = {}, []
built = [m for m in list_months(OUT_P["features"]) if marker(m)]
if not built:
    blocks.append(N("No month built yet."))
else:
    lm = built[-1]
    X = spark.read.parquet(month_dir(lm))
    full = X.where(F.col("window_months") == REC_WINDOW)
    cov = X.agg(F.count("*").alias("clients"),
                F.avg((F.col("own_known_n") > 0).cast("double")).alias("own account known"),
                F.avg((F.col("own_known_self_n") > 0).cast("double")).alias("known from self-pay"),
                F.avg(((F.col("own_ext_n_in") + F.col("own_ext_n_out")) > 0).cast("double")).alias("third-party flow via own account"),
                F.avg(((F.col("mig_n_in") + F.col("mig_n_out")) > 0).cast("double")).alias("migrated partner"),
                F.avg((F.col("rb_n_out") + F.col("rb_n_in") > 0).cast("double")).alias("broken standing relationship"),
                F.avg((F.col("nbr_dark_n") > 0).cast("double")).alias("dark PNC counterparty"),
                F.avg(F.col("padj_out").isNotNull().cast("double")).alias("padj_out defined")).toPandas()
    pc = [c for c in cov.columns if c != "clients"]
    tot = X.agg(*[F.sum(c).alias(c) for c in ["rb_n_out", "rb_obs_n_out", "rb_alive_n_out", "rb_n_in", "rb_obs_n_in",
                                              "rb_alive_n_in", "own_ext_amt_in", "own_ext_amt_out", "mig_amt_in",
                                              "mig_amt_out", "own_ext_name_only_amt_in", "own_ext_name_only_amt_out"]]).first()
    br = pd.DataFrame([{"side": d, "broken": tot[f"rb_n_{d}"], "observable": tot[f"rb_obs_n_{d}"],
                        "observable share": (tot[f"rb_obs_n_{d}"] or 0) / max(tot[f"rb_n_{d}"] or 0, 1),
                        "alive": tot[f"rb_alive_n_{d}"],
                        "alive share of observable": (tot[f"rb_alive_n_{d}"] or 0) / max(tot[f"rb_obs_n_{d}"] or 0, 1),
                        "own-account USD m": float(tot[f"own_ext_amt_{d}"] or 0) / 1e6,
                        "of which migrated USD m": float(tot[f"mig_amt_{d}"] or 0) / 1e6,
                        "of which name-match only USD m": float(tot[f"own_ext_name_only_amt_{d}"] or 0) / 1e6} for d in D])
    RES.update({"P1_own_known": float(cov["own account known"].iloc[0]),
                "P2_obs_share": float(br.loc[br.side == "out", "observable share"].iloc[0]),
                "P3_alive_share": float(br.loc[br.side == "out", "alive share of observable"].iloc[0])})
    look = ["own_known_n", "own_ext_n_in", "mig_n_in", "mig_share_in", "std_n_out", "rb_n_out", "rb_alive_share_out",
            "std_obs_amt_share_out", "own_chg_out", "partner_chg_out", "padj_out", "padj_in", "nbr_dark_amt_share"]
    qs = full.select(*[F.expr(f"percentile_approx({c}, array(0.1, 0.5, 0.9))").alias(f"q_{c}") for c in look],
                     *[F.avg(F.col(c).isNull().cast("double")).alias(f"n_{c}") for c in look],
                     *[F.avg((F.col(c) != 0).cast("double")).alias(f"p_{c}") for c in look]).first()
    dist = pd.DataFrame([{"feature": c, "p10": (qs[f"q_{c}"] or [None] * 3)[0], "p50": (qs[f"q_{c}"] or [None] * 3)[1],
                          "p90": (qs[f"q_{c}"] or [None] * 3)[2], "null_share": qs[f"n_{c}"], "nonzero_share": qs[f"p_{c}"]}
                         for c in look])
    topm = (X.where((F.col("mig_n_in") + F.col("mig_n_out")) > 0)
             .select("customer_name", "customer_pwr_id", "int_partners_n_in", "mig_n_in", (F.col("mig_amt_in") / 1e3).alias("mig_in_usd_k"),
                     "own_ext_top_fi_in", "int_partners_n_out", "mig_n_out", (F.col("mig_amt_out") / 1e3).alias("mig_out_usd_k"),
                     "own_ext_top_fi_out", "own_known_self_n", "own_known_n")
             .orderBy(F.desc(F.col("mig_n_in") + F.col("mig_n_out")), F.desc("mig_in_usd_k")).limit(15).toPandas())
    dest = (X.select(F.explode(F.array(*[F.struct(F.lit(d).alias("side"), F.col(f"own_ext_top_fi_{d}").alias("bank"),
                                                  F.col(f"own_ext_amt_{d}").cast("double").alias("usd"),
                                                  (F.col(f"mig_n_{d}") > 0).cast("int").alias("mig")) for d in D])).alias("x"))
             .select("x.*").where(F.col("bank").isNotNull())
             .groupBy("bank").agg(F.count("*").alias("client_sides"), F.sum("mig").alias("with_migrated_partner"),
                                  (F.sum("usd") / 1e6).alias("usd_m"))
             .orderBy(F.desc("usd_m")).limit(15).toPandas())
    # own-name accounts with the most third parties this month: check that client and account names are the same company
    _FPC.clear(); _SLIM_OK.clear(); _SELFP_OK.clear()
    rg = own_registry(lm).where((F.col("n_fam") == 1) & ~F.col("hub_drop"))
    tope = (spark.read.parquet(month_in("pairs", lm)).where(F.col("scope") == "ext").select("cust", "other", "amount")
              .join(rg.select("pwr", "other", "self_ev", "name_ev", "exact", "tier"), "other", "inner")
              .join(CMAP.select(F.col("mdm_id").alias("cust"), F.col("pwr").alias("a_pwr")), "cust", "left")
              .where(~F.coalesce(F.col("a_pwr") == F.col("pwr"), F.lit(False)))
              .groupBy("pwr", "other", "self_ev", "name_ev", "exact", "tier")
              .agg(F.countDistinct("cust").alias("third_parties"), (F.sum("amount") / 1e3).cast("double").alias("usd_k"))
              .orderBy(F.desc("third_parties"), F.desc("usd_k")).limit(15)
              .join(PWR.select("pwr", "customer_name"), "pwr", "left")
              .join(spark.read.parquet(month_in("cpm", lm)).select(F.col("ent_id").alias("other"),
                    (F.col("ent_name") if CPM_OPT["ent_name"] else F.lit(None).cast("string")).alias("account_name")), "other", "left")
              .select(F.col("customer_name").alias("client"), "account_name", "self_ev", "name_ev", "exact", "tier",
                      "third_parties", "usd_k", F.col("pwr").alias("customer_pwr_id"), F.col("other").alias("ent_id"))
              .orderBy(F.desc("third_parties"), F.desc("usd_k")).toPandas())
    # monthly series over every built month
    ser = (spark.read.parquet(OUT_P["features"]).groupBy("month").agg(
              F.count("*").alias("clients"),
              F.avg((F.col("own_known_n") > 0).cast("double")).alias("own_known_share"),
              F.avg(((F.col("mig_n_in") + F.col("mig_n_out")) > 0).cast("double")).alias("migrated_share"),
              (F.sum(F.col("own_ext_amt_in") + F.col("own_ext_amt_out")) / 1e6).cast("double").alias("own_ext_usd_m"),
              F.sum("rb_n_out").alias("broken_out"),
              (F.sum("rb_obs_n_out") / F.sum("rb_n_out")).alias("observable_share_out"),
              (F.sum("rb_alive_n_out") / F.sum("rb_obs_n_out")).alias("alive_share_out"),
              F.expr("percentile_approx(padj_out, 0.5)").alias("padj_out_median"),
              F.avg((F.col("nbr_dark_n") > 0).cast("double")).alias("dark_nbr_share"))
            .orderBy("month").toPandas())
    ser.to_csv(OUTD / "4_monthly_series.csv", index=False)
    fig, ax = plt.subplots(2, 3, figsize=(14, 6.5))
    for a, (c, ttl) in zip(ax.ravel(), [("clients", "Clients with a row"), ("own_known_share", "Own account known"),
                                        ("migrated_share", "Migrated partner"), ("observable_share_out", "Broken, paid side: observable"),
                                        ("alive_share_out", "Observable broken: alive"), ("padj_out_median", "padj_out median")]):
        a.plot(ser.month, ser[c], marker="o", ms=3, color="#1d4e89"); a.set_title(ttl, fontsize=10)
        a.tick_params(axis="x", labelrotation=90, labelsize=7); a.grid(alpha=0.3)
    fig.suptitle("Counterparty features by month", fontsize=12); fig.tight_layout()
    chart = savefig(fig, "4_monthly_series")
    blocks += [H(f"QA · latest month {lm}"),
               TB("Coverage (share of clients with a row)", cov, pct=tuple(pc)),
               TB("Broken standing relationships and own-account dollars", br, pct=("observable share", "alive share of observable")),
               TB(f"Distributions (clients with the full {REC_WINDOW}-month window)", dist, pct=("null_share", "nonzero_share")),
               TB("Clients with the most migrated partners (check by name)", topm),
               TB("Own-name accounts with the most third parties (client and account should be the same company)", tope),
               N("self_ev = seen in the client's own self-pay; name_ev = exact name match; exact = account name key equals "
                 f"the client's; tier = hub_tier (above {OWN_MAX_HUB_TIER} only exact names are kept)."),
               TB("Banks holding the clients' own-name accounts that receive or send the third-party flow", dest),
               TB("Monthly series", ser, pct=("own_known_share", "migrated_share", "observable_share_out", "alive_share_out", "dark_nbr_share")),
               N(f"Chart saved: {chart}")]
show(*blocks)

## §5 Early signal read against the m4 exit labels

Before model 5, a direct read of whether these features carry exit signal. Population as in m4: client-months where
the money is still here (`live` and `msh` in `pkg_attr_cust_month`), with the outcome window fully observed. Event =
the earlier of full exit and balance drain (`ab_t`) within `HORIZON` months. Unconfirmed full exits are left out.

- **Univariate AUC** of each feature, and the exit-rate lift when a count is above zero. AUCs come from all exits and
  a random sample of stayer months. This is a ranking read, not a model: m5 measures what the features add to m4.
- **Event-time profile:** the share of leavers showing each signal by month before exit, against stayers.
- **Competitor or contraction:** each leaver's last 6 months before exit, classified by the counterparty evidence,
  with the same classes for stayer months as the baseline. Counted in clients and in balance dollars.

In [ ]:
# §5  Early signal read
blocks = []
READ = ["own_known_n", "own_ext_n_in", "own_ext_n_out", "own_ext_amt_in", "mig_n_in", "mig_n_out", "mig_share_in",
        "mig_share_out", "rb_n_out", "rb_n_in", "rb_alive_n_out", "rb_alive_n_in", "rb_shrunk_n_out", "rb_shrunk_n_in",
        "rb_alive_share_out",
        "rb_alive_share_in", "rb_alive_amt_share_out", "std_obs_amt_share_out", "own_chg_out", "own_chg_in",
        "partner_chg_out", "partner_chg_in", "padj_out", "padj_in", "nbr_dark_n", "nbr_dark_amt_share"]
COUNTS = {"own_ext_n_in", "own_ext_n_out", "mig_n_in", "mig_n_out", "rb_n_out", "rb_n_in", "rb_alive_n_out",
          "rb_alive_n_in", "rb_shrunk_n_out", "rb_shrunk_n_in", "nbr_dark_n", "own_known_n"}
if not SIGNAL_READ:
    blocks.append(N("SIGNAL_READ = False"))
elif not (table_exists(T_LBL_MONTH) and table_exists(T_LBL_DIM)) or not built:
    blocks.append(BAD(f"Skipped: {T_LBL_MONTH} / {T_LBL_DIM} not found, or no feature month built."))
else:
    LMc, LDc = set(spark.table(T_LBL_MONTH).columns), set(spark.table(T_LBL_DIM).columns)
    miss = sorted({"cust_pwr_id", "t", "live", "msh"} - LMc) + sorted({"cust_pwr_id", "ab_t", "exit_t", "provisional"} - LDc)
    if miss:
        raise RuntimeError(f"label tables lack {miss}")
    M0 = ym_idx(LBL_START)
    t_col = lambda c: (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - 1 - F.lit(M0)).cast("int")
    if "ym" in LMc:
        bad_ym = spark.table(T_LBL_MONTH).where(t_col("ym") != F.col("t")).limit(1).count()
        if bad_ym:
            raise RuntimeError(f"pkg_attr_cust_month: t does not match ym with LBL_START = {LBL_START}")
    FT = spark.table(OUT["features"])
    # power-id format: raw or leading zeros stripped, whichever matches more ids
    kl, kf = spark.table(T_LBL_MONTH).select("cust_pwr_id").distinct(), FT.select("customer_pwr_id").distinct()
    rates = {}
    for mode in ("raw", "strip"):
        a = kl.select(pk("cust_pwr_id", mode).alias("k")).distinct()
        b = kf.select(pk("customer_pwr_id", mode).alias("k")).distinct()
        rates[mode] = (a.join(b, "k").count(), a.count())
    KEY_MODE = max(rates, key=lambda x: rates[x][0])
    dollars = "bal_med_live" in LDc
    LM = spark.table(T_LBL_MONTH).select(pk("cust_pwr_id", KEY_MODE).alias("k"), F.col("t").cast("int").alias("t"), "live", "msh")
    LD = spark.table(T_LBL_DIM).select(pk("cust_pwr_id", KEY_MODE).alias("k"), F.col("ab_t").cast("int").alias("ab_t"),
                                       F.col("exit_t").cast("int").alias("exit_t"), F.col("provisional").cast("int").alias("prov"),
                                       (F.col("bal_med_live").cast("double") if dollars else F.lit(None).cast("double")).alias("bal"))
    unconf = (F.col("ab_t").isNotNull() & (F.col("prov") == 1) & (F.col("ab_t") == F.col("exit_t")))
    END_T = int(LM.agg(F.max("t")).first()[0])
    FK = FT.select(pk("customer_pwr_id", KEY_MODE).alias("k"), t_col("month").alias("t"), *READ)
    pop = (LM.where((F.col("live") == 1) & (F.col("msh") == 1) & (F.col("t") <= END_T - HORIZON))
             .join(LD, "k", "left").where(F.col("ab_t").isNull() | (F.col("t") < F.col("ab_t")))
             .withColumn("y", (F.col("ab_t").isNotNull() & (F.col("ab_t") <= F.col("t") + HORIZON)).cast("int"))
             .where(~((F.col("y") == 1) & unconf))
             .join(FK.withColumn("vis", F.lit(1)), ["k", "t"], "left")
             .withColumn("vis", F.coalesce("vis", F.lit(0))).persist(MD))
    ps = pop.groupBy("vis").agg(F.count("*").alias("client_months"), F.countDistinct("k").alias("clients"),
                                F.sum("y").alias("exits_within_h"), F.sum("bal").alias("bal")).toPandas()
    ps["exit_rate"] = ps.exits_within_h / ps.client_months
    ps["vis"] = ps.vis.map({1: "features present", 0: "no feature row (no payments in the window)"})
    vis = pop.where(F.col("vis") == 1)
    n_pos, n_neg = vis.agg(F.sum("y"), F.sum(1 - F.col("y"))).first()
    n_pos, n_neg = int(n_pos or 0), int(n_neg or 0)
    rate = min(1.0, MAX_NEG_SAMPLE / max(n_neg, 1))
    P = (vis.where((F.col("y") == 1) | (F.rand(seed=13) < rate))
            .withColumn("w", F.when(F.col("y") == 1, 1.0).otherwise(1.0 / rate))
            .select("k", "t", "y", "w", *READ).toPandas())
    base_rate = n_pos / max(n_pos + n_neg, 1)

    def auc(score, y):
        m = score.notna()
        s, yy = score[m], y[m]
        npos = int(yy.sum()); nneg = len(yy) - npos
        if npos == 0 or nneg == 0:
            return np.nan
        r = s.rank(method="average")
        return float((r[yy == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg))
    rows = []
    for f in READ:
        x = P[f].astype(float); w = P.w
        a = auc(x, P.y)
        row = {"feature": f, "coverage": float((w * x.notna()).sum() / w.sum()),
               "nonzero": float((w * (x.fillna(0) != 0)).sum() / w.sum()), "AUC": a,
               "direction": ("higher = riskier" if a >= 0.5 else "lower = riskier") if not np.isnan(a) else "—",
               "AUC (best direction)": max(a, 1 - a) if not np.isnan(a) else np.nan}
        if f in COUNTS:
            on, off = x.fillna(0) > 0, x.fillna(0) == 0
            r_on = float((w[on] * P.y[on]).sum() / w[on].sum()) if w[on].sum() else np.nan
            r_off = float((w[off] * P.y[off]).sum() / w[off].sum()) if w[off].sum() else np.nan
            row.update({"exit rate if > 0": r_on, "exit rate if 0": r_off, "lift": r_on / r_off if r_off else np.nan})
        rows.append(row)
    AU = pd.DataFrame(rows).sort_values("AUC (best direction)", ascending=False)
    AU.to_csv(OUTD / "5a_univariate.csv", index=False)
    mig_any = (P.mig_n_in.fillna(0) + P.mig_n_out.fillna(0)) > 0
    lift_of = lambda on: float((P.w[on] * P.y[on]).sum() / P.w[on].sum()) / float((P.w[~on] * P.y[~on]).sum() / P.w[~on].sum())
    RES.update({"P4_mig_lift": lift_of(mig_any) if mig_any.any() else np.nan,
                "P5_padj_out_auc": 1 - auc(P.padj_out.astype(float), P.y) if P.padj_out.notna().any() else np.nan,
                "P6_alive_share_auc": auc(P.rb_alive_share_out.astype(float), P.y),
                "P9_dark_lift": lift_of(P.nbr_dark_n.fillna(0) > 0) if (P.nbr_dark_n.fillna(0) > 0).any() else np.nan})

    # ---- event-time profile: leavers by month before exit vs stayers -------------------------------------------------
    PROF = {"mig_n_in": "migrated payer", "own_ext_n_in": "third party pays own account elsewhere",
            "rb_alive_n_out": "broken, partner alive (paid side)", "rb_shrunk_n_out": "broken, partner shrinking (paid side)",
            "nbr_dark_n": "dark PNC counterparty"}
    LV = LD.where(F.col("ab_t").isNotNull() & ~unconf).select("k", "ab_t")
    prof = (FK.join(LV, "k").withColumn("rel", F.col("t") - F.col("ab_t")).where(F.col("rel").between(-12, -1))
              .groupBy("rel").agg(F.count("*").alias("rows"), *[F.avg((F.col(f) > 0).cast("double")).alias(f) for f in PROF],
                                  F.expr("percentile_approx(padj_out, 0.5)").alias("padj_out_median"))
              .orderBy("rel").toPandas())
    stay = (FK.join(LD.where(F.col("ab_t").isNull()).select("k"), "k")
              .agg(F.count("*").alias("rows"), *[F.avg((F.col(f) > 0).cast("double")).alias(f) for f in PROF],
                   F.expr("percentile_approx(padj_out, 0.5)").alias("padj_out_median")).toPandas())
    stay.insert(0, "rel", "stayers")
    PR = pd.concat([prof.astype({"rel": str}), stay], ignore_index=True)
    PR.to_csv(OUTD / "5b_event_time_profile.csv", index=False)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, (f, lab) in enumerate(PROF.items()):
        cc = ["#1d4e89", "#f28c28", "#12a39a", "#d1495b", "#8d99ae"][i]
        ax.plot(prof.rel, prof[f], marker="o", ms=3, label=lab, color=cc)
        ax.axhline(float(stay[f].iloc[0]), ls=":", lw=1, color=cc)
    ax.set_xlabel("months before exit"); ax.set_ylabel("share of leavers"); ax.grid(alpha=0.3)
    ax.legend(fontsize=8); ax.set_title("Counterparty signals before exit (dotted: stayers)", fontsize=11)
    ch1 = savefig(fig, "5b_event_time_profile")

    # ---- competitor or contraction: last 6 months before exit -------------------------------------------------------
    w6 = Window.partitionBy("k").orderBy("t").rangeBetween(-5, 0)
    TR = FK.select("k", "t",
                   F.max(F.col("mig_n_in") + F.col("mig_n_out")).over(w6).alias("mig6"),
                   F.sum(F.col("rb_alive_n_in") + F.col("rb_alive_n_out")).over(w6).alias("alive6"),
                   F.sum(F.col("rb_shrunk_n_in") + F.col("rb_shrunk_n_out")).over(w6).alias("shrunk6"))
    for c in ("mig6", "alive6", "shrunk6"):
        TR = TR.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
    cls = (F.when(F.col("t").isNull(), "6 not visible the month before exit")
            .when(F.col("mig6") > 0, "1 moved: partners follow the client to its own account elsewhere")
            .when(F.col("alive6") > F.col("shrunk6"), "2 displacement-like: broken with partners still active at PNC")
            .when(F.col("shrunk6") > F.col("alive6"), "3 contraction-like: partners shrinking at PNC too")
            .when(F.col("alive6") > 0, "4 mixed")
            .otherwise("5 no counterparty evidence"))
    lv = (LV.join(LD.select("k", "bal"), "k", "left").withColumn("t_m1", F.col("ab_t") - 1)
            .join(TR.withColumnRenamed("t", "t_m1").withColumn("t", F.col("t_m1")), ["k", "t_m1"], "left")
            .withColumn("class", cls).groupBy("class").agg(F.count("*").alias("leavers"), (F.sum("bal") / 1e6).alias("leaver_bal_usd_m"))
            .toPandas())
    sm_ = spark.createDataFrame(P.loc[P.y == 0, ["k", "t"]].astype({"t": "int32"}))
    stc = (sm_.join(TR, ["k", "t"], "left").withColumn("class", cls).groupBy("class").agg(F.count("*").alias("stayer_months")).toPandas())
    CL = lv.merge(stc, on="class", how="outer").sort_values("class").fillna(0)
    vis_lv = CL.loc[~CL["class"].str.startswith("6"), "leavers"].sum()
    CL["leaver share"] = CL.leavers / max(CL.leavers.sum(), 1)
    CL["leaver $ share"] = CL.leaver_bal_usd_m / max(CL.leaver_bal_usd_m.sum(), 1e-9)
    CL["stayer-month share"] = CL.stayer_months / max(CL.stayer_months.sum(), 1)
    CL.to_csv(OUTD / "5c_competitor_or_contraction.csv", index=False)
    g = lambda s: float(CL.loc[CL["class"].str.startswith(s), "leavers"].sum())
    RES.update({"P7_disp_vs_contr": (g("1") + g("2"), g("3")), "P8_mig_leavers": g("1") / max(vis_lv, 1)})
    pop.unpersist()
    blocks += [H(f"Signal read · horizon {HORIZON} months · power-id key {KEY_MODE} "
                 f"(ids matched raw {rates['raw'][0]:,} / strip {rates['strip'][0]:,} of {rates['raw'][1]:,})"),
               TB("Population: money still here, outcome window observed", ps, pct=("exit_rate",)),
               N(f"Exit rate among client-months with features: {base_rate:.2%} ({n_pos:,} exits, {n_neg:,} stayer months; "
                 f"stayers sampled at {rate:.1%} for the AUCs, weighted back for rates and lifts)."),
               TB("Univariate read, best first", AU, pct=("coverage", "nonzero", "exit rate if > 0", "exit rate if 0")),
               TB("Share of leavers with each signal, by month before exit (stayers in the last row)", PR,
                  pct=tuple(PROF)),
               TB("Leavers' last 6 months: competitor or contraction (stayer months as the baseline)", CL,
                  pct=("leaver share", "leaver $ share", "stayer-month share")),
               N(f"Charts and CSVs in {OUTD.resolve()}: {pathlib.Path(ch1).name}, 5a_univariate.csv, "
                 "5b_event_time_profile.csv, 5c_competitor_or_contraction.csv")]
show(*blocks)

## §6 Predictions, scored mechanically

In [ ]:
# §6  Predictions
def verdict(v, ok):
    return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else ("✓ hit" if ok(v) else "✗ miss")
p7 = RES.get("P7_disp_vs_contr")
PRED = [("P1", "≥ 5% of clients have a known own-name account elsewhere", "60%", RES.get("P1_own_known"), lambda v: v >= 0.05, "pct"),
        ("P2", "≥ 30% of broken standing relationships (paid side) are observable", "50%", RES.get("P2_obs_share"), lambda v: v >= 0.30, "pct"),
        ("P3", "≥ 50% of observable broken relationships are with live partners", "65%", RES.get("P3_alive_share"), lambda v: v >= 0.50, "pct"),
        ("P4", "migrated partner: exit-rate lift ≥ 3", "55%", RES.get("P4_mig_lift"), lambda v: v >= 3, "num"),
        ("P5", "padj_out AUC (lower = riskier) ≥ 0.60", "45%", RES.get("P5_padj_out_auc"), lambda v: v >= 0.60, "num"),
        ("P6", "rb_alive_share_out AUC (higher = riskier) ≥ 0.58", "50%", RES.get("P6_alive_share_auc"), lambda v: v >= 0.58, "num"),
        ("P7", "leavers: displacement-like + moved > contraction-like", "60%",
         None if p7 is None else p7[0] - p7[1], lambda v: v > 0, "num"),
        ("P8", "≥ 10% of visible leavers show a migrated partner in the last 6 months", "35%", RES.get("P8_mig_leavers"), lambda v: v >= 0.10, "pct"),
        ("P9", "dark PNC counterparty: exit-rate lift ≥ 1.5", "40%", RES.get("P9_dark_lift"), lambda v: v >= 1.5, "num")]
tab = pd.DataFrame([{"#": p, "prediction": txt, "confidence": c,
                     "observed": ("—" if v is None or (isinstance(v, float) and np.isnan(v)) else (f"{v:.1%}" if kind == "pct" else f"{v:,.3f}")),
                     "verdict": verdict(v, ok)} for p, txt, c, v, ok, kind in PRED])
hits = (tab.verdict == "✓ hit").sum(); scored = (tab.verdict != "—").sum()
show(H(f"Predictions: {hits} of {scored} scored hit"), TB("Pre-registered predictions", tab),
     N("P7 observed = (moved + displacement-like) − contraction-like leavers."))

## §7 Impala

Run once per new or re-created table: `INVALIDATE METADATA`. After a later run that only added months: `REFRESH`.

In [ ]:
# §7  Impala statements
show(TB("Impala", pd.DataFrame({"statement": [f"INVALIDATE METADATA {OUT[k]};" for k in ("features", "dict")]
                                              + [f"REFRESH {OUT[k]};" for k in ("features", "dict")]})))